In [ ]:
#importing packages
import pandas as pd
import folium
import geopandas as gp
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

#setting pandas columns so I can see all of them
pd.set_option('display.max_columns', None)

In [ ]:
#reading in data
cejst = pd.read_csv("CEJST_2.0-communities.csv", low_memory = False)
atlas = pd.read_csv('im3_open_source_data_center_atlas_v2026.02.09.csv')
fractracker = pd.read_csv("U.S. Data Centers - FracTracker.csv")

In [ ]:
#cleaning fractracker

#setting numeric columns; removing commas
#can check if this worked with "fractracker.dtypes"
numeric_columns = ["MW (high)", "MW (low)", "Number of generators", "Number of buildings"]
for column in numeric_columns:
    fractracker[column] = pd.to_numeric(fractracker[column].str.replace(',', ''), errors='coerce')

In [ ]:
#cleaning cejst

cejst = cejst.rename(columns={'Census tract 2010 ID':'GEOID'})
cejst = cejst.rename(columns={"State/Territory":"State"})

In [ ]:
#exploring atlas

#looking at Ohio
atlas[atlas.state == "Ohio"]

In [ ]:
#exploring fractracker

#making an Ohio dataset
fractracker_ohio = fractracker[fractracker.State == "OH"]

#making a dataset of proposed data centers in Ohio
fractracker_prop_ohio = fractracker_ohio[fractracker_ohio.Status == "Proposed"].copy()

#ok, it looks like atlas has more data of operating data centers. However, fracTracker has proposed data centers.
#I want to delve into how Prometheus compares to all other data centers. Is it the biggest? In the top 10? etc.
#potential analysis could be seeing how many of these proposed centers are in vulnerable census tracts.
#scope could be US if I want something general, or Ohio if I want to get more into the weeds.

#ok, looks like the "location confidence" of these data centers is mostly high.
#let's look at prometheus now.

In [ ]:
#looking for prometheus

fractracker_MW = fractracker.sort_values(by='MW (high)', ascending=False).head(40)

#It's the 37th biggest data center... according to fractracker!

In [ ]:
#does atlas say the same thing?

#oop... atlas doesn't even have MW

#what about square footage?
atlas_sqft = atlas.sort_values(by='sqft', ascending=False).head(40)

#hmm... prometheus doesn't show up on here. what does fractracker say?

In [ ]:
fractracker_sqft = fractracker.sort_values(by='Facility size (sq ft)', ascending=False).head(40)

#fractracker has prometheus as the #1 top spot for square footage -- this seemed suspsicious so I did further investigating
#the number they provide is incorrect so I submitted an update suggestion to fractracker

In [ ]:
#we should probably check if atlas even has prometheus at all


#making an Ohio dataset
atlasOhio = atlas[atlas.state == "Ohio"]

atlasOhio[atlasOhio.operator == "Meta"]

#aah... it considers Prometheus to be 3 different data centers. good to know.

In [ ]:
#let's get started by mapping the data centers

ohio_map = folium.Map(location=[40, -83], zoom_start=7)
for idx, datacenter in fractracker_prop_ohio.iterrows():
    folium.Marker([datacenter['Lat'], datacenter['Long']],
                            popup=datacenter['Name'],
                             icon=folium.Icon(color='purple')).add_to(ohio_map)
ohio_map

#ok wonderful. now how do I add the counties and cejst to this? shade them by level of vulnerability?

In [ ]:
#ok, I think (?) that "Census tract 2010 ID" is the same as GEOID from the shapefile. hopefully. 
#the values are mostly 11 numbers but a few have 10 numbers.
#let's try it. 

In [ ]:
#reading in census tracts shapefile
#btw I am using https://programminghistorian.org/en/lessons/choropleth-maps-python-folium 
#basically walking through these steps

# Read the shapefile
census_tracts = gp.read_file("tl_2025_39_tract.zip")

# Print the first few rows of the GeoDataFrame
#print(census_tracts.head(10))

#what percentage of cejst has geoid?
#print(cejst['GEOID'].notna().sum())
#cejst['GEOID'].notna().sum() / len(cejst)
#100%!
#I like this code... gonna save it in case I want it later

In [ ]:
#cleaning census_tracts before converting it to json

#this is to remove the lakes

census_tracts = census_tracts[census_tracts['ALAND'] > 1_000_000]

In [ ]:
#trying to figure out what geojson is and how to get the census tracts to match up in choropleth key_on
geojson = census_tracts.to_json()
print(census_tracts.columns.tolist())


In [ ]:
#trying to figure out what's the deal with some of cejst's GEOID's being less than 11 digits

#census_tracts.crs
count = (cejst['GEOID'].astype(str).str.len() != 11).sum()


count_b = (cejst['GEOID'].astype(str).str.len() == 11).sum()

count_c = (cejst['GEOID'].astype(str).str.len() == 10).sum()

#print(count, count_b, count_c)

cejst_ohio = cejst[cejst.State == "Ohio"]

#cejst_ohio[cejst_ohio['GEOID'].astype(str).str.len() == 10]

#TLDR; there are some GEOID's (about 1 in every 7) that are 10 digits instead of 11. However, this is not a problem in Ohio.

In [ ]:
#ff_df['points'] = gp.points_from_xy(ff_df.longitude, ff_df.latitude, crs="EPSG:4326")
#cejst_ohio = gp.GeoDataFrame(data=cejst_ohio,geometry='GEOID')

In [ ]:


#it seems like the lake counties are not coming from cejst but from geojson



In [ ]:
#making map of census tracts in ohio colored by PM2.5 concentration


#initial map of Ohio
def initial_map():
    tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}'
    attr = 'Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ, TomTom, Intermap, iPC, USGS, FAO, NPS, NRCAN, GeoBase, Kadaster NL, Ordnance Survey, Esri Japan, METI, Esri China (Hong Kong), and the GIS User Community'

    center = [40,-83]


    map = folium.Map(location=center,
                zoom_start = 7,
                tiles = tiles,
                attr = attr)
    return map

base_map = initial_map()


#adding color based on PM2.5
folium.Choropleth(
    geo_data = geojson,
    data = cejst_ohio,
    key_on = 'feature.properties.GEOID',        
    columns = ['GEOID','PM2.5 in the air'],
    bins = 9,
    fill_color = 'OrRd',
    fill_opacity = 0.8,
    line_opacity = 0.2,
    nan_fill_color = 'grey',
    legend_name = 'PM2.5 in the air by census tract'
    ).add_to(base_map)

base_map

In [ ]:
#adding proposed data centers to the map

for idx, datacenter in fractracker_prop_ohio.iterrows():
    folium.Marker([datacenter['Lat'], datacenter['Long']],
                            popup=datacenter['Name'],
                             icon=folium.Icon(color='blue')).add_to(base_map)
base_map


In [ ]:
#ok we have the map. wonderful!
#I think the next step is that I want to make some sort of linear plot looking at the relationship between proposed data centers and pm2.5
#might have to do that matching thing that historian guy was talking about because I don't have the GEOID on fractracker, but I do have lat/long
#still using https://programminghistorian.org/en/lessons/choropleth-maps-python-folium for this

#gotta combine fractracker_prop_ohio with census_tracts and cejst so that I have the PM2.5 data connected to the data centers

#first I need to combine the census_tracts dataset to the fractracker_prop_ohio dataset 
#census_tracts.crs

#Location stuff that allows it to match with the geodata
fractracker_prop_ohio['Points'] = gp.points_from_xy(fractracker_prop_ohio.Long, fractracker_prop_ohio.Lat, crs="EPSG:4269")
fractracker_prop_ohio = gp.GeoDataFrame(data=fractracker_prop_ohio,geometry='Points')

fractracker_combined = gp.sjoin(left_df = fractracker_prop_ohio,
                  right_df = census_tracts,
                  how = 'left')

#fractracker_prop_ohio.info()
#yay it worked!

In [ ]:
#another join to combine the PM2.5 data

#first have to do this so they are both int64
fractracker_combined["GEOID"] = pd.to_numeric(fractracker_combined["GEOID"])

In [ ]:
fractracker_combined = pd.merge(
    fractracker_combined, cejst, how="left", on=["GEOID"])

In [ ]:
#histogram time

#making a df just for the histogram
hist_df = fractracker_combined[["GEOID", "PM2.5 in the air"]].copy()


# Plotting a histogram
plt.hist(hist_df["PM2.5 in the air"], bins=10, edgecolor='black')

# Adding labels and title
plt.xlabel('PM2.5 Concentration')
plt.ylabel('# of Proposed Data Centers')
plt.title('Proposed Data Centers by PM2.5 Concentration')

# Display the plot
plt.show()

In [ ]:
cejst["PM2.5 in the air"].dtype

In [ ]:
#pre_affected_rows = cejst[cejst['GEOID'].isin(affected_tracts)]

#pre_affected_rows

In [ ]:
#ok let's do some analysis
#arbitrarily let's say that PM2.5 increases by 0.01 micromules wherever there is a proposed data center
#Let's figure out how to represent that.
#fractracker_combined was a left join which means it doesn't have all of the other counties that didn't have proposed data centers.
#first let's get the following data: the GEOID of each census tract in fractracker_combined and how many data centers it has

#making a list of GEOID's where the proposed data centers are
affected_tracts = fractracker_combined["GEOID"].tolist()

#new dataset
cejst_model = cejst.copy()

#locating the affected tracts, and adding 0.01 to PM2.5 column
cejst_model.loc[cejst_model['GEOID'].isin(affected_tracts), 'PM2.5 in the air'] += 0.01

#making a new column for affected or not affected
cejst_model["Affected"] = cejst_model['GEOID'].isin(affected_tracts)

#Looking at affected ones
cejst_model[cejst_model["Affected"] == True]

In [ ]:
#doing a chi squared test to compare percentages. the following code is from Claude so be aware.


tracts_affected = cejst_model[cejst_model["Affected"] == True]
tracts_not_affected = cejst_model[cejst_model["Affected"] == False]

#Create contingency table
contingency_table = pd.crosstab(cejst_model['Affected'], 
                                cejst_model['Identified as disadvantaged'])

#Perform chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("\nPercentage of affected census tracts that are disadvantaged:", tracts_affected["Identified as disadvantaged"].sum()/len(tracts_affected))
print("Percentage of unaffected census tracts that are disadvantaged:", tracts_not_affected["Identified as disadvantaged"].sum()/len(tracts_not_affected))

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of freedom: {dof}")

if p_value < 0.05:
    print("\nResult: Statistically significant association (p < 0.05)")
else:
    print("\nResult: No statistically significant association (p >= 0.05)")

In [ ]:
#ok now let's do the same thing but with national data instead of ohio data

'''

from pygris import tracts
from shapely.geometry import Point


#making global proposed dataset
fractracker_prop = fractracker[fractracker.Status == "Proposed"].copy()

#cejst global dataset
cejst_global = cejst.copy()

#combining datasets
#Location stuff that allows it to match with the geodata
#I have to do it differently this time because it's all US not just Ohio

# List of all state FIPS codes (as strings)
states = [
    '01', '02', '04', '05', '06', '08', '09', '10', '11', '12', '13', '15', '16', 
    '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', 
    '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', 
    '44', '45', '46', '47', '48', '49', '50', '51', '53', '54', '55', '56', '72'
]

# Download and concatenate all state tracts
all_tracts = []
for state in states:
    try:
        state_tracts = tracts(state=state, year=2021, cb=True)
        all_tracts.append(state_tracts)
        print(f"Downloaded {state}")
    except:
        print(f"Failed for {state}")
        
tract_geometries = gp.GeoDataFrame(pd.concat(all_tracts, ignore_index=True))

# Merge with your data
cejst_global = cejst_global.merge(
    tract_geometries[['GEOID', 'geometry']], 
    on='GEOID',
    how='left'
)

cejst_global = gp.GeoDataFrame(cejst_global, geometry='geometry', crs=tract_geometries.crs)

# Create data center points
fractracker_points = gp.GeoDataFrame(
    fractracker_prop,
    geometry=gp.points_from_xy(fractracker_prop['Long'], fractracker_prop['Lat']),
    crs='EPSG:4326'
)

# Match CRS and do spatial join
fractracker_points = fractracker_points.to_crs(cejst_global.crs)




#cejst_global['data center'] = ~gp.sjoin(
#    cejst_global,
##    fractracker_points,
 #   how='left',
 #   predicate='contains'
#)['index_right'].isna()


#fractracker_prop['Points'] = gp.points_from_xy(fractracker_prop.Long, fractracker_prop.Lat, crs="EPSG:4269")
#fractracker_prop = gp.GeoDataFrame(data=fractracker_prop,geometry='Points')

#fractracker_combined_global = gp.sjoin(left_df = fractracker_prop,
#                  right_df = census_tracts,
#                  how = 'left')

#another join to combine the PM2.5 data



#merge fractracker and cejst
fractracker_combined_global = pd.merge(
    fractracker_prop, cejst_global, how="left", on=["GEOID"])


#making a list of GEOID's where the proposed data centers are
affected_tracts = fractracker_combined["GEOID"].tolist()



#making a new column for affected or not affected
cejst_model_global["Affected"] = cejst_model_global['GEOID'].isin(affected_tracts)

#Looking at affected ones
#cejst_model_global[cejst_model_global["Affected"] == True]


#doing a chi squared test to compare percentages. the following code is from Claude so be aware.

tracts_affected = cejst_model_global[cejst_model_global["Affected"] == True]
tracts_not_affected = cejst_model_global[cejst_model_global["Affected"] == False]

#Create contingency table
contingency_table = pd.crosstab(cejst_model_global['Affected'], 
                                cejst_model_global['Identified as disadvantaged'])

#Perform chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("\nPercentage of affected census tracts that are disadvantaged:", tracts_affected["Identified as disadvantaged"].sum()/len(tracts_affected))
print("Percentage of unaffected census tracts that are disadvantaged:", tracts_not_affected["Identified as disadvantaged"].sum()/len(tracts_not_affected))

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of freedom: {dof}")

if p_value < 0.05:
    print("\nResult: Statistically significant association (p < 0.05)")
else:
    print("\nResult: No statistically significant association (p >= 0.05)")

'''


In [ ]:
#that was a mess. I asked Claude to clean it up. Let's see if THIS works.

# Import necessary libraries
from pygris import tracts
import geopandas as gp
import pandas as pd
from scipy.stats import chi2_contingency

# --- STEP 1: Prepare fractracker data ---
fractracker_prop = fractracker[fractracker.Status == "Proposed"].copy()

# --- STEP 2: Download all US census tracts ---
states = [
    '01', '02', '04', '05', '06', '08', '09', '10', '11', '12', '13', '15', '16',
    '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29',
    '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42',
    '44', '45', '46', '47', '48', '49', '50', '51', '53', '54', '55', '56', '72'
]

all_tracts = []
for state in states:
    try:
        state_tracts = tracts(state=state, year=2021, cb=True)
        all_tracts.append(state_tracts)
        print(f"Downloaded {state}")
    except Exception as e:
        print(f"Failed for {state}: {e}")

tract_geometries = gp.GeoDataFrame(pd.concat(all_tracts, ignore_index=True))

# --- STEP 3: Prepare CEJST data with geometry ---
cejst_global = cejst.copy()

# Fix GEOID data types to match - convert both to strings
cejst_global['GEOID'] = cejst_global['GEOID'].astype(str)
tract_geometries['GEOID'] = tract_geometries['GEOID'].astype(str)

# Merge CEJST with tract geometries
cejst_global = cejst_global.merge(
    tract_geometries[['GEOID', 'geometry']],
    on='GEOID',
    how='left'
)
cejst_global = gp.GeoDataFrame(cejst_global, geometry='geometry', crs=tract_geometries.crs)

# --- STEP 4: Create point geometries for data centers ---
fractracker_points = gp.GeoDataFrame(
    fractracker_prop,
    geometry=gp.points_from_xy(fractracker_prop['Long'], fractracker_prop['Lat']),
    crs='EPSG:4326'
)

# Match CRS
fractracker_points = fractracker_points.to_crs(cejst_global.crs)

# --- STEP 5: Spatial join to find which tract each data center is in ---
fractracker_with_tracts = gp.sjoin(
    fractracker_points,
    cejst_global,
    how='left',
    predicate='within'  # 'within' is better than 'contains' when joining points to polygons
)

# --- STEP 6: Create the combined dataset ---
# This keeps all fractracker data and adds the CEJST columns
fractracker_combined_global = fractracker_with_tracts.copy()

# --- STEP 7: Mark affected tracts in CEJST ---
# Get list of unique GEOIDs where data centers are located
affected_tracts = fractracker_combined_global["GEOID"].dropna().unique().tolist()

# Create analysis dataset from cejst_global (not undefined cejst_model_global)
cejst_model_global = cejst_global.copy()
cejst_model_global["Affected"] = cejst_model_global['GEOID'].isin(affected_tracts)

# --- STEP 8: Chi-squared test ---
tracts_affected = cejst_model_global[cejst_model_global["Affected"] == True]
tracts_not_affected = cejst_model_global[cejst_model_global["Affected"] == False]

# Create contingency table
contingency_table = pd.crosstab(
    cejst_model_global['Affected'],
    cejst_model_global['Identified as disadvantaged']
)

print("\nContingency Table:")
print(contingency_table)

# Perform chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# Results
print("\n--- RESULTS ---")
print(f"Number of affected tracts: {len(tracts_affected)}")
print(f"Number of unaffected tracts: {len(tracts_not_affected)}")
print(f"\nPercentage of affected census tracts that are disadvantaged: {tracts_affected['Identified as disadvantaged'].sum()/len(tracts_affected)*100:.2f}%")
print(f"Percentage of unaffected census tracts that are disadvantaged: {tracts_not_affected['Identified as disadvantaged'].sum()/len(tracts_not_affected)*100:.2f}%")
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Degrees of freedom: {dof}")

if p_value < 0.05:
    print("\nResult: Statistically significant association (p < 0.05)")
else:
    print("\nResult: No statistically significant association (p >= 0.05)")

# Optional: Save the results
print(f"\n--- DATA CENTER SUMMARY ---")
print(f"Total proposed data centers: {len(fractracker_prop)}")
print(f"Data centers matched to census tracts: {fractracker_combined_global['GEOID'].notna().sum()}")
print(f"Data centers not matched: {fractracker_combined_global['GEOID'].isna().sum()}")